# **Project Name**    -  Netflix clustering project




##### **Project Type**    - Unsupervised
##### **Contribution**    - Individual


# **Project Summary -**

This project focuses on performing exploratory data analysis (EDA) and clustering on a dataset of Netflix movies and TV shows to uncover meaningful patterns and group similar content. With the rapid growth of streaming platforms, understanding content distribution and user preferences has become essential for improving recommendation systems and enhancing user experience. The primary objective of this project is to analyze Netflix’s content library and identify natural clusters within the data using unsupervised learning techniques.

The dataset used in this project contains information about movies and TV shows available on Netflix, including attributes such as type (Movie or TV Show), title, director, cast, country, release year, rating, duration, and genre. Initially, the dataset was explored to understand its structure, data types, and the presence of missing values. Data cleaning and preprocessing steps were carried out to improve data quality. Missing values in columns such as director, cast, and country were handled by replacing them with “Unknown,” while missing values in the rating column were filled using the most frequent value. Duplicate records were also removed to ensure accuracy.

Exploratory Data Analysis (EDA) was conducted to gain insights into the dataset. Various visualizations were created to analyze the distribution of content across different dimensions. It was observed that movies dominate the platform compared to TV shows, indicating a higher volume of movie content. Countries like the United States and India were identified as major contributors to the dataset. Genre analysis revealed that Drama and Comedy are among the most popular categories. Additionally, a noticeable increase in content production was observed after 2015, reflecting the rapid expansion of streaming services.
To prepare the data for clustering, feature engineering techniques were applied. Since machine learning models require numerical input, categorical variables such as type, rating, and genre were converted into numerical form using Label Encoding. A new feature representing the primary genre was created by extracting it from the listed_in column. The selected features were then scaled using StandardScaler to ensure that all variables contributed equally to the clustering process.

The clustering results revealed clear patterns in the data. Each cluster represented a group of similar content, such as movies, TV shows, or specific genre-based categories. These groupings can be useful for improving recommendation systems by suggesting similar content to users based on their viewing preferences.

In conclusion, this project demonstrates how data analysis and clustering techniques can be used to organize and understand large content datasets. The insights gained can help improve content recommendations, enhance user experience, and support better business decisions for streaming platforms like Netflix.

# **GitHub Link -**

https://github.com/Sandhiya-003/Netflix_clustering

# **Problem Statement**


The objective of this project is to group Netflix movies and TV shows into meaningful clusters based on their features, in order to identify similar content and support better content recommendations.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger') # Added for POS tagging

# Initialize lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
import scipy.stats as stats
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

### Dataset Loading

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('netflix_titles.csv')
df.head()

### Dataset First View

In [ ]:
df.head()

### Dataset Rows & Columns count

In [ ]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

### Dataset Information

In [ ]:
df.describe()

#### Duplicate Values

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

#### Missing Values/Null Values

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
sns.heatmap(df.isnull(), cbar=False)
plt.show()

### What did you know about your dataset?

The dataset contains information about Netflix movies and TV shows, including details such as type (movie or TV show), title, director, cast, country, release year, rating, duration, and genre.

It includes both categorical and numerical features, with some missing values present in columns like director, cast, and country.

The dataset provides a comprehensive view of Netflix content, which can be used to analyze patterns and group similar content using clustering techniques.



## ***2. Understanding Your Variables***

In [ ]:
df.columns

In [ ]:
df.describe()

### Variables Description

In [ ]:
df.info()

### Variables Description

- **show_id**: (Object) Unique ID for each show. No missing values.
- **type**: (Object) Type of content, either 'Movie' or 'TV Show'. No missing values.
- **title**: (Object) Title of the show/movie. No missing values.
- **director**: (Object) Director(s) of the content. Contains missing values.
- **cast**: (Object) Main actors/actresses in the content. Contains missing values.
- **country**: (Object) Country where the content was produced. Contains missing values.
- **date_added**: (Object) Date when the content was added to Netflix. Contains missing values.
- **release_year**: (Int64) Year the content was originally released. No missing values.
- **rating**: (Object) Content maturity rating. Contains missing values.
- **duration**: (Object) Duration of the content (e.g., '93 min' for movies, '4 Seasons' for TV shows). No missing values.
- **listed_in**: (Object) Genres/categories the content is listed under. No missing values.
- **description**: (Object) A brief description of the content. No missing values.

### Check Unique Values for each variable.

In [ ]:
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].nunique())
    print(df[col].unique()[:10])
    print("-"*50)

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)
df['rating'].fillna(df['rating'].mode()[0], inplace=True)

# Convert date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Extract year and month from date_added
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# Extract main genre (first genre)
df['main_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])

# Clean duration (optional improvement)
df['duration'] = df['duration'].astype(str)

df.head()

### What all manipulations have you done and insights you found?

In this dataset, several preprocessing steps were performed. Duplicate records were removed to ensure data quality. Missing values in important columns such as director, cast, and country were handled by replacing them with "Unknown", while missing values in the rating column were filled using the most frequent value.

The date_added column was converted into datetime format, and new features such as year_added and month_added were extracted to enable time-based analysis.

Additionally, the listed_in column containing multiple genres was simplified by extracting the primary genre into a new column called main_genre.

These transformations made the dataset cleaner and more suitable for analysis and clustering, improving the overall quality of insights derived from the data.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
df['type'].value_counts().plot(kind='bar')

##### 1. Why did you pick the specific chart?

**To understand the distribution of contents**

##### 2. What is/are the insight(s) found from the chart?

**Movies are more than tv shows**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Helps identify need to increase TV Shows for engagement.**

**Negative: Too many movies may reduce binge-watching retention.**

#### Chart - 2

In [ ]:
df['country'].value_counts().head(10).plot(kind='bar')

##### 1. Why did you pick the specific chart?

**To identify top content producing countries**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Focus on strong markets.**

**Negative: Lack of diversity from other regions.**

#### Chart - 3

In [ ]:
df['release_year'].value_counts().sort_index().plot()

##### 1. Why did you pick the specific chart?

**To know about the content growth**

##### 2. What is/are the insight(s) found from the chart?

**Content increased after 2015**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Shows platform growth.**

**Negative: Rapid growth may affect quality.**

#### Chart - 4

In [ ]:
df['rating'].value_counts().plot(kind='bar')
plt.title("Rating Distribution")
plt.show()

##### 1. Why did you pick the specific chart?

**To understand targeting audience.**

##### 2. What is/are the insight(s) found from the chart?

**TV-MA and TV-14 dominate.**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Strong adult audience targeting.**

**Negative: Less content for kids.**

#### Chart - 5

In [ ]:
df['duration'].value_counts().head(10).plot(kind='bar')

##### 1. Why did you pick the specific chart?

**To analyze movie length**

##### 2. What is/are the insight(s) found from the chart?

**Most movies are in standard length**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Helps optimize duration.**

#### Chart - 6

In [ ]:
df['main_genre'].value_counts().head(10).plot(kind='bar')
plt.title("Top Genres")
plt.show()

##### 1. Why did you pick the specific chart?

**To identify popular genres.**


##### 2. What is/are the insight(s) found from the chart?

**Dramas and comedy dominate**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Focus on popular genres.**

**Negative: Less genre diversity.**

#### Chart - 7

In [ ]:
sns.countplot(x='release_year', hue='type', data=df)
plt.xticks(rotation=90)
plt.show()

##### 1. Why did you pick the specific chart?

**To compare trends over time**

##### 2. What is/are the insight(s) found from the chart?

**Movies dominate across years**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Strong movie production.**

**Negative: Less TV show growth.**

#### Chart - 8

In [ ]:
df['director'].value_counts().head(10).plot(kind='bar')
plt.title("Top Directors")
plt.show()

##### 1. Why did you pick the specific chart?

**To identify frequent contributors**

##### 2. What is/are the insight(s) found from the chart?

**Few directors dominate.**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Helps partnerships.**

**Negative: Less diversity in creators.**

#### Chart - 9

In [ ]:
df['cast'].str.split(',').explode().value_counts().head(10).plot(kind='bar')
plt.title("Top Actors")
plt.show()

##### 1. Why did you pick the specific chart?

**To identify popular actors**

##### 2. What is/are the insight(s) found from the chart?

**Some actors appear frequently**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Helps marketing.**

**Negative: Repetition reduces novelty.**

#### Chart - 10

In [ ]:
df['main_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])
sns.countplot(y='main_genre', hue='type', data=df, order=df['main_genre'].value_counts().index[:10])

##### 1. Why did you pick the specific chart?

**To analyze how genre relates to audience rating**

##### 2. What is/are the insight(s) found from the chart?

**Certain genres are more adult oriented**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive: Better targeting.**

**Negative: Limited audience coverage.**

#### Chart - 11

In [ ]:
pd.crosstab(df['main_genre'], df['rating']).plot(kind='bar', stacked=True)
plt.show()

##### 1. Why did you pick the specific chart?

**To analyze genre vs audience.**

##### 2. What is/are the insight(s) found from the chart?

**Certain genre target specific age group.**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Improves target recommendation.**

#### Chart - 12

In [ ]:
df['year_added'].value_counts().sort_index().plot()
plt.show()

##### 1. Why did you pick the specific chart?

**To see Netflix growth**

##### 2. What is/are the insight(s) found from the chart?

**Content addition increased over time**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Shows expansion strategy**

#### Chart - 13

In [ ]:
df['cluster'].value_counts().plot(kind='bar')
plt.title("Cluster Distribution")
plt.show()

##### 1. Why did you pick the specific chart?

**To understand the clustering result**

##### 2. What is/are the insight(s) found from the chart?

**Data is grouped into clusters**

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Will help in recommendation systems**

#### Chart - 14 - Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type_enc'] = le.fit_transform(df['type'])
df['rating_enc'] = le.fit_transform(df['rating'])
df['genre_enc'] = le.fit_transform(df['main_genre'])

import seaborn as sns
sns.heatmap(df[['type_enc','rating_enc','genre_enc']].corr(), annot=True)

##### 1. Why did you pick the specific chart?

**To find relationships**

##### 2. What is/are the insight(s) found from the chart?

**Negative correlation exists**

#### Chart - 15 - Pair Plot

In [ ]:
sns.pairplot(df[['type_enc','rating_enc','genre_enc']])

##### 1. Why did you pick the specific chart?

**To visualize relationships**

##### 2. What is/are the insight(s) found from the chart?

**Patterns and clusters are visible**

## ***5. Hypothesis Testing***

### Hypothetical Statement - 1

Movies have a significantly higher average release year than TV Shows on Netflix,
meaning Netflix has been adding newer movies compared to TV Shows.

#### 1. Null & Alternate Hypothesis
H₀ (Null)      : There is no significant difference in average release_year
                  between Movies and TV Shows.
                  
H₁ (Alternate) : Movies have a significantly higher average release_year
                  than TV Shows.

#### 2. Perform an appropriate statistical test.

In [ ]:
movies_year  = df[df['type'] == 'Movie']['release_year'].dropna()
tvshow_year  = df[df['type'] == 'TV Show']['release_year'].dropna()

# Independent two-sample t-test (Welch's t-test, equal_var=False)
t_stat, p_value = stats.ttest_ind(movies_year, tvshow_year, equal_var=False)

print("=" * 55)
print("Hypothesis 1 — Movies vs TV Shows : Release Year")
print("=" * 55)
print(f"Movies  mean release year  : {movies_year.mean():.2f}")
print(f"TV Shows mean release year : {tvshow_year.mean():.2f}")
print(f"T-statistic : {t_stat:.4f}")
print(f"P-value     : {p_value:.6f}")
if p_value < 0.05:
    print("Result: Reject H₀ — Significant difference exists.")
else:
    print("Result: Fail to reject H₀ — No significant difference.")

##### Which statistical test have you done to obtain P-Value?

Independent two-sample t-test (Welch's variant).

##### Why did you choose the specific statistical test?

release_year is a continuous numerical variable, and we are comparing means
between two independent groups (Movies vs TV Shows). Welch's t-test is chosen
because the two groups may have unequal variances and different sample sizes.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

The distribution of content ratings (TV-MA, TV-14, etc.) differs significantly
between Movies and TV Shows — i.e., content type and rating are NOT independent.

#### 1. Null & Alternate Hypothesis
H₀ (Null)      : Content type (Movie/TV Show) and rating are independent.

H₁ (Alternate) : Content type and rating are significantly associated.

#### 2. Perform an appropriate statistical test.

In [ ]:
contingency_table = pd.crosstab(df['type'], df['rating'])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency_table)
print("\n" + "=" * 55)
print("Hypothesis 2 — Content Type vs Rating Independence")
print("=" * 55)
print(f"Chi-Square Statistic : {chi2:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-value              : {p_chi:.6f}")
if p_chi < 0.05:
    print("Result: Reject H₀ — Type and Rating are NOT independent.")
else:
    print("Result: Fail to reject H₀ — Type and Rating are independent.")

##### Which statistical test have you done to obtain P-Value?

Chi-square test of Independence

##### Why did you choose the specific statistical test?

Both variables (type and rating) are categorical. The chi-square test checks
whether there is a statistically significant association between two categorical
variables using their observed vs expected frequency counts.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

he United States produces significantly more content per year on average
than India on Netflix.

#### 1. Null & Alternate Hypothesis
H₀ (Null)      : There is no significant difference in the number of titles
                  produced per year between the USA and India.
H₁ (Alternate) : The USA produces significantly more content per year than India.

#### 2. Perform an appropriate statistical test.

In [ ]:
usa_per_year  = df[df['country'].str.contains('United States', na=False)].groupby('release_year').size()
india_per_year = df[df['country'].str.contains('India', na=False)].groupby('release_year').size()

u_stat, p_mw = stats.mannwhitneyu(usa_per_year, india_per_year, alternative='greater')

print("\n" + "=" * 55)
print("Hypothesis 3 — USA vs India : Titles per Year")
print("=" * 55)
print(f"USA   mean titles/year   : {usa_per_year.mean():.2f}")
print(f"India mean titles/year   : {india_per_year.mean():.2f}")
print(f"Mann-Whitney U statistic : {u_stat:.4f}")
print(f"P-value                  : {p_mw:.6f}")
if p_mw < 0.05:
    print("Result: Reject H₀ — USA produces significantly more content per year.")
else:
    print("Result: Fail to reject H₀ — No significant difference.")

##### Which statistical test have you done to obtain P-Value?

Man-Whitney U test

##### Why did you choose the specific statistical test?

The distribution of titles per year is not guaranteed to be normal (small
sample, skewed counts). The Mann-Whitney U test is a non-parametric alternative
to the t-test that does not assume normality — ideal for comparing two
independent groups on an ordinal or non-normal continuous variable.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
print("\n--- Missing Values after wrangling ---")
print(df[['director','cast','country','rating','release_year']].isnull().sum())

# Impute any remaining NaN in year_added with median
df['release_year'].fillna(df['release_year'].median(), inplace=True)

#### What all missing value imputation techniques have you used and why did you use those techniques?

- **Mode imputation** for 'rating' — rating is categorical; the mode (most
  frequent value) is the best representative replacement.
- **Constant 'Unknown'** for director/cast/country — preserving the row while
  signalling missing data prevents information loss.
- **Median imputation** for 'year_added' — median is robust to skew/outliers,
  suitable for year-based numerical columns.

### 2. Handling Outliers

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.boxplot(x=df['release_year'], color='skyblue')
plt.title('Boxplot — release_year (before outlier treatment)')
plt.tight_layout()
plt.savefig('boxplot_release_year.png', dpi=100)
plt.show()

# IQR-based clipping for release_year
Q1 = df['release_year'].quantile(0.25)
Q3 = df['release_year'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['release_year'] = df['release_year'].clip(lower=lower_bound, upper=upper_bound)

print(f"\nrelease_year clipped to [{lower_bound:.0f}, {upper_bound:.0f}]")

##### What all outlier treatment techniques have you used and why did you use those techniques?

Since the dataset mainly contains categorical data, outliers are minimal. Therefore, no major outlier treatment was required.

### 3. Categorical Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Ensure 'main_genre' is created if it was missing due to execution order
# This line is from the data wrangling section (cell wk-9a2fpoLcV)
if 'main_genre' not in df.columns:
    df['main_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])

df['type_enc'] = le.fit_transform(df['type'])
df['rating_enc'] = le.fit_transform(df['rating'])
df['genre_enc'] = le.fit_transform(df['main_genre'])

#### What all categorical encoding techniques have you used & why did you use those techniques?

Label Encoding was used to convert categorical variables into numerical format, as clustering algorithms require numerical input.

### 4. Textual Data Preprocessing

#### 1. Expand Contraction

In [ ]:
import re
contractions_dict = {
    "can't":"cannot", "won't":"will not", "don't":"do not",
    "it's":"it is", "i'm":"i am", "they're":"they are",
    "he's":"he is", "she's":"she is", "we're":"we are",
    "i've":"i have", "you've":"you have", "isn't":"is not",
    "aren't":"are not", "wasn't":"was not", "weren't":"were not"
}
def expand_contractions(text):
    for contraction, expansion in contractions_dict.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)
    return text

df['desc_clean'] = df['description'].fillna('').apply(expand_contractions)

#### 2. Lower Casing

In [ ]:
df['desc_clean'] = df['desc_clean'].str.lower()

#### 3. Removing Punctuations

In [ ]:
df['desc_clean'] = df['desc_clean'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
df['desc_clean'] = df['desc_clean'].apply(lambda x: re.sub(r'http\S+', '', x))
df['desc_clean'] = df['desc_clean'].apply(lambda x: re.sub(r'\b\w*\d\w*\b', '', x))

#### 5. Removing Stopwords & Removing White spaces

In [ ]:
def remove_stopwords_func(text):
    tokens = text.split()
    return ' '.join([w for w in tokens if w not in stop_words])

In [ ]:
df['desc_clean'] = df['desc_clean'].apply(remove_stopwords_func)
df['desc_clean'] = df['desc_clean'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

#### 6. Rephrase Text

In [ ]:
df['desc_clean'] = df['desc_clean'].apply(
    lambda x: re.sub(r'\b(\w+)( \1\b)+', r'\1', x))

#### 7. Tokenization

In [ ]:
df['tokens'] = df['desc_clean'].apply(lambda x: x.split())

#### 8. Text Normalization

In [ ]:
def lemmatize_text(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

df['tokens_lemma'] = df['tokens'].apply(lemmatize_text)
df['desc_lemma']   = df['tokens_lemma'].apply(lambda x: ' '.join(x))

##### Which text normalization technique have you used and why?

**Lemmatization** (via NLTK WordNetLemmatizer).
Lemmatization was chosen over stemming because it returns real dictionary words
(e.g., 'running' → 'run', not 'runn'), producing cleaner and more interpretable
TF-IDF features for content-based clustering.

#### 9. Part of speech tagging

In [ ]:
nltk.download('averaged_perceptron_tagger_eng')
sample_tokens = df['tokens_lemma'].iloc[0]
pos_tags = pos_tag(sample_tokens)
print("\nSample POS Tags:", pos_tags[:10])

#### 10. Text Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=100, min_df=2)
tfidf_matrix = tfidf.fit_transform(df['desc_lemma'])
print(f"\nTF-IDF matrix shape : {tfidf_matrix.shape}")

##### Which text vectorization technique have you used and why?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.sparse import hstack, csr_matrix

# Prepare structured numerical and encoded categorical features for scaling
# Assuming 'release_year', 'year_added', 'type_enc', 'rating_enc', 'genre_enc' are ready from previous steps

# IMPUTATION FOR year_added - This is the fix for the NaN issue
# Fill any remaining NaN in year_added with its median value
df['year_added'].fillna(df['year_added'].median(), inplace=True)

structured_features_df = df[['release_year', 'year_added', 'type_enc', 'rating_enc', 'genre_enc']].copy()

# Scale the structured features
schaler = StandardScaler()
scaled_structured_features = scaler.fit_transform(structured_features_df)

# Combine scaled structured features with the TF-IDF matrix
# TF-IDF matrix ('tfidf_matrix') is generated in a prior step 'yBRtdhth6JDE'
# Convert scaled_structured_features to sparse format for consistent hstacking
scaled_features = hstack([csr_matrix(scaled_structured_features), tfidf_matrix])

# Apply KMeans clustering to create the 'cluster' column for visualization purposes.
# Using n_clusters=5 as determined to be optimal later in the notebook (ML Model 1).
# random_state is set for reproducibility.
kmeans_initial = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['cluster'] = kmeans_initial.fit_predict(scaled_features.toarray()) # fit_predict expects dense array if not sparse-aware

print(f"Shape of combined_features for clustering: {scaled_features.shape}")
print(f"Unique clusters created: {df['cluster'].nunique()}")
print(f"First 5 rows of df['cluster']:\n{df['cluster'].head()}")

**TF-IDF (Term Frequency–Inverse Document Frequency)**.
TF-IDF was chosen because it weights terms by how unique they are across
all documents, reducing the influence of very common words and highlighting
terms that are distinctive to each title — ideal for content similarity
clustering. max_features=100 keeps dimensionality manageable.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# The 'scaled_features' created in the previous cell (61a20a0e) already combines
# the scaled structured features and the TF-IDF matrix. So, we will use it directly.
# No additional 'combined_features' creation is needed here.

#### 2. Feature Selection

In [ ]:
print(f"\nCombined feature matrix shape : {scaled_features.shape}")

##### What all feature selection methods have you used  and why?

- **Structured features selected**: release_year, year_added, type_enc,
  rating_enc, genre_enc — these directly describe content characteristics
  and are the most relevant for grouping similar titles.
- **Text features**: Top 100 TF-IDF terms from the description column —
  descriptions capture thematic similarity between titles.
- **Dropped**: show_id (identifier), title (text ID), date_added (redundant
  after extracting year/month), director/cast/country (too sparse after
  encoding, and would blow up dimensionality without benefit).

##### Which all features you found important and why?

- **type_enc** — strongest separator (Movie vs TV Show).
- **genre_enc** — content category is the primary clustering dimension.
- **rating_enc** — captures audience target group.
- **release_year** — separates classic content from modern.
- **TF-IDF description** — enables semantic similarity within clusters.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

No log or power transformation needed. All numerical features (release_year,
year_added) are already on a linear, interpretable scale. Transformation
would not improve Euclidean-distance-based KMeans.

### 6. Data Scaling

In [ ]:
print("\nScaled features (first 2 rows):")
print(scaled_features[:2])


##### Which method have you used to scale you data and why?

Method: StandardScaler (Z-score normalisation).
KMeans uses Euclidean distances, so features with larger ranges (e.g.,
release_year in the 1920–2021 range) would dominate over binary-encoded
features without scaling. StandardScaler centres each feature at mean=0
with std=1, giving every feature equal weight in the distance calculation.

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes, dimensionality reduction is beneficial here:
1. The combined feature matrix (structured + TF-IDF) has 105 dimensions.
2. PCA reduces it to 2–3 components for visualisation and speeds up clustering.
3. PCA also removes multicollinearity between TF-IDF terms.

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(scaled_features)

print(f"\nPCA Explained Variance Ratio : {pca.explained_variance_ratio_}")
print(f"Total variance explained     : {pca.explained_variance_ratio_.sum():.2%}")

# Visualise PCA-reduced clusters
plt.figure(figsize=(9,6))
scatter = plt.scatter(pca_coords[:,0], pca_coords[:,1],
                      c=df['cluster'], cmap='tab10', alpha=0.5, s=10)
plt.colorbar(scatter, label='Cluster')
plt.title('PCA — 2D Cluster Visualisation (KMeans, k=5)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.savefig('pca_cluster_viz.png', dpi=100)
plt.show()

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

PCA was selected because it is the standard linear technique for reducing
structured/numerical feature spaces. It maximises retained variance along
orthogonal axes, which is compatible with the Euclidean distances that KMeans
relies on. n_components=2 was chosen for interpretable 2D visualisation.

### 8. Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(scaled_features, test_size=0.2, random_state=42)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"\nTrain size : {X_train.shape[0]} rows")
print(f"Test  size : {X_test.shape[0]} rows")

##### What data splitting ratio have you used and why?

Split ratio: 80% Train / 20% Test.
80/20 is the standard split. The training set is used to fit all clustering
models; the test set is used to evaluate how well the learned cluster
centroids generalise to unseen data (via silhouette score on test predictions).

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

This is an unsupervised clustering task — there are no class labels, so
traditional class imbalance (SMOTE, undersampling) does not apply.

In [ ]:
print("\nCluster size distribution:")
print(df['cluster'].value_counts().sort_index())

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

If cluster sizes are severely imbalanced (e.g., one cluster with >70% of data),
we can try:
- Adjusting K upward so over-dense clusters are split further.
- Using DBSCAN (density-based), which does not force equal cluster sizes.
Both alternatives are implemented in Section 7 (Models 2 & 3).

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
wcss       = []
sil_scores = []
k_range    = range(2, 11)

# Convert X_train to a dense array for KMeans, as some metrics expect it.
X_train_dense = X_train.toarray()

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = km.fit_predict(X_train_dense) # Use dense array here
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_train_dense, labels, sample_size=min(2000, len(X_train_dense)), random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), wcss, marker='o', color='steelblue')
axes[0].set_title('Elbow Method — WCSS vs K')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('WCSS')
axes[0].grid(True)

axes[1].plot(list(k_range), sil_scores, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].grid(True)

plt.suptitle('KMeans — Optimal K Selection', fontsize=14)
plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=100)
plt.show()

optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++',
                       random_state=42, n_init=10)
kmeans_final.fit(X_train_dense)
train_labels = kmeans_final.labels_
test_labels  = kmeans_final.predict(X_test.toarray()) # Use dense array for prediction

# ── Evaluation Metrics ───────────────────────────────────────
sil_train = silhouette_score(X_train_dense, train_labels, sample_size=min(2000, len(X_train_dense)), random_state=42)
sil_test  = silhouette_score(X_test.toarray(),  test_labels,  sample_size=min(1000, len(X_test.toarray())), random_state=42)
db_train  = davies_bouldin_score(X_train_dense, train_labels)
ch_train  = calinski_harabasz_score(X_train_dense, train_labels)

print("\n" + "=" * 55)
print("Model 1 — KMeans (k=5) Evaluation")
print("=" * 55)
print(f"Silhouette Score  (Train) : {sil_train:.4f}")
print(f"Silhouette Score  (Test)  : {sil_test:.4f}")
print(f"Davies-Bouldin Score      : {db_train:.4f}  (lower = better)")
print(f"Calinski-Harabasz Score   : {ch_train:.4f}  (higher = better)")

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
metrics     = ['Silhouette\n(Train)', 'Silhouette\n(Test)', 'Davies-Bouldin\n(lower=better)', 'Calinski-Harabasz\n(÷1000)']
values      = [sil_train, sil_test, db_train, ch_train / 1000]
colors      = ['steelblue', 'lightblue', 'tomato', 'mediumseagreen']

plt.figure(figsize=(10, 5))
bars = plt.bar(metrics, values, color=colors, edgecolor='black', width=0.5)
plt.title('KMeans (k=5) — Evaluation Metric Scores', fontsize=14)
plt.ylabel('Score')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('kmeans_eval_chart.png', dpi=100)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from itertools import product

k_options    = [4, 5, 6]
init_options = ['k-means++', 'random']
best_score   = -1
best_params  = {}

print("\nKMeans Hyperparameter Search:")
print("-" * 45)

X_train_dense = X_train.toarray()
X_test_dense  = X_test.toarray()

for k, init in product(k_options, init_options):
    km_cv = KMeans(n_clusters=k, init=init, random_state=42, n_init=10)
    labels_cv = km_cv.fit_predict(X_train_dense)
    score = silhouette_score(X_train_dense, labels_cv, sample_size=min(2000, len(X_train_dense)), random_state=42)
    print(f"  k={k}, init={init:<12} → Silhouette = {score:.4f}")
    if score > best_score:
        best_score  = score
        best_params = {'n_clusters': k, 'init': init}

print(f"\nBest params : {best_params}  (Silhouette = {best_score:.4f})")

kmeans_tuned = KMeans(**best_params, random_state=42, n_init=10)
kmeans_tuned.fit(X_train_dense)
tuned_train_labels = kmeans_tuned.labels_
tuned_test_labels  = kmeans_tuned.predict(X_test_dense)

sil_tuned = silhouette_score(X_test_dense, tuned_test_labels, sample_size=min(1000, len(X_test_dense)), random_state=42)
db_tuned  = davies_bouldin_score(X_train_dense, tuned_train_labels)

print(f"\nTuned KMeans — Test Silhouette : {sil_tuned:.4f}")
print(f"Tuned KMeans — Davies-Bouldin  : {db_tuned:.4f}")
labels_cmp  = ['Silhouette (before)', 'Silhouette (after)', 'DB Score (before)', 'DB Score (after)']
values_cmp  = [sil_test, sil_tuned, db_train, db_tuned]
colors_cmp  = ['lightcoral', 'steelblue', 'lightcoral', 'steelblue']

plt.figure(figsize=(10, 5))
bars = plt.bar(labels_cmp, values_cmp, color=colors_cmp, edgecolor='black', width=0.5)
plt.title('KMeans — Before vs After Hyperparameter Tuning', fontsize=13)
plt.ylabel('Score')
for bar, val in zip(bars, values_cmp):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('kmeans_before_after_tuning.png', dpi=100)
plt.show()

##### Which hyperparameter optimization technique have you used and why?

Grid Search over (n_clusters, init).
We manually iterated over a parameter grid and selected the combination
with the highest Silhouette Score on the training set.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Switching from the default k=5 to the best-found parameters (printed above)
shows an improvement in Silhouette Score, confirming that even small changes
in K or initialisation method can influence cluster quality.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

sample_idx   = np.random.choice(len(X_train_dense), size=200, replace=False)
X_sample     = X_train_dense[sample_idx]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90,
           show_leaf_counts=True)
plt.title('Dendrogram — Agglomerative Clustering (Ward Linkage, Sample=200)')
plt.xlabel('Data Point Index')
plt.ylabel('Euclidean Distance')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=100)
plt.show()

# ── Fit Agglomerative Model ──────────────────────────────────
agg = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
agg_train_labels = agg.fit_predict(X_train_dense)

# Predict on test: assign each test point to nearest train centroid
from sklearn.metrics import pairwise_distances_argmin_min
# Compute cluster centroids from training
train_centroids = np.array([X_train_dense[agg_train_labels == k].mean(axis=0)
                             for k in range(optimal_k)])
agg_test_labels, _ = pairwise_distances_argmin_min(X_test_dense, train_centroids)
sil_agg_train = silhouette_score(X_train_dense, agg_train_labels, sample_size=min(2000, len(X_train_dense)), random_state=42)
sil_agg_test  = silhouette_score(X_test_dense,  agg_test_labels,  sample_size=min(1000, len(X_test_dense)), random_state=42)
db_agg        = davies_bouldin_score(X_train_dense, agg_train_labels)
ch_agg        = calinski_harabasz_score(X_train_dense, agg_train_labels)

print("\n" + "=" * 55)
print("Model 2 — Agglomerative Clustering (k=5, Ward)")
print("=" * 55)
print(f"Silhouette Score  (Train) : {sil_agg_train:.4f}")
print(f"Silhouette Score  (Test)  : {sil_agg_test:.4f}")
print(f"Davies-Bouldin Score      : {db_agg:.4f}")
print(f"Calinski-Harabasz Score   : {ch_agg:.4f}")

# ── Evaluation Metric Chart ───────────────────────────────────
metrics_agg = ['Silhouette\n(Train)', 'Silhouette\n(Test)', 'Davies-Bouldin\n(lower=better)', 'Calinski-H.\n(÷1000)']
values_agg  = [sil_agg_train, sil_agg_test, db_agg, ch_agg / 1000]
colors_agg  = ['mediumpurple', 'plum', 'tomato', 'mediumseagreen']

plt.figure(figsize=(10, 5))
bars = plt.bar(metrics_agg, values_agg, color=colors_agg, edgecolor='black', width=0.5)
plt.title('Agglomerative Clustering — Evaluation Metric Scores', fontsize=14)
plt.ylabel('Score')
for bar, val in zip(bars, values_agg):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('agg_eval_chart.png', dpi=100)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
linkage_options  = ['ward', 'average', 'complete']
k_options_agg    = [4, 5, 6]
best_agg_score   = -1
best_agg_params  = {}

print("\nAgglomerative Hyperparameter Search:")
print("-" * 45)

X_train_dense = X_train.toarray()

for k, lnk in product(k_options_agg, linkage_options):
    try:
        agg_cv = AgglomerativeClustering(n_clusters=k, linkage=lnk)
        labels_agg_cv = agg_cv.fit_predict(X_train_dense)
        score = silhouette_score(X_train_dense, labels_agg_cv, sample_size=min(2000, len(X_train_dense)), random_state=42)
        print(f"  k={k}, linkage={lnk:<10} → Silhouette = {score:.4f}")
        if score > best_agg_score:
            best_agg_score  = score
            best_agg_params = {'n_clusters': k, 'linkage': lnk}
    except Exception as e:
        print(f"  k={k}, linkage={lnk} → Error: {e}")

print(f"\nBest Agglomerative params : {best_agg_params}  (Silhouette = {best_agg_score:.4f})")

# Re-fit with best params
agg_tuned        = AgglomerativeClustering(**best_agg_params)
agg_tuned_labels = agg_tuned.fit_predict(X_train_dense)
sil_agg_tuned    = silhouette_score(X_train_dense, agg_tuned_labels, sample_size=min(2000, len(X_train_dense)), random_state=42)
db_agg_tuned     = davies_bouldin_score(X_train_dense, agg_tuned_labels)

print(f"Tuned Agglomerative — Silhouette : {sil_agg_tuned:.4f}")
print(f"Tuned Agglomerative — DB Score   : {db_agg_tuned:.4f}")

##### Which hyperparameter optimization technique have you used and why?

Grid Search over (n_clusters, linkage).
By testing all combinations of K and linkage method and selecting on Silhouette
Score, we find the configuration that produces the most cohesive and
well-separated clusters.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The best linkage method (typically 'ward') matches or improves upon the default.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

- **Silhouette Score (range -1 to 1)**: Measures how similar a content item
  is to its own cluster vs other clusters. A higher score means the
  recommendation engine will suggest more relevant and similar titles to users,
  directly improving user satisfaction and watch time.
- **Davies-Bouldin Score (lower = better)**: Measures ratio of intra-cluster
  scatter to inter-cluster separation. A low DB score means clusters are
  compact and far apart — the recommendation system won't accidentally suggest
  content from unrelated clusters (e.g., children's shows to adult thriller fans).
- **Calinski-Harabasz Score (higher = better)**: Ratio of between-cluster
  dispersion to within-cluster dispersion. A high CH score means the platform
  can define distinct content categories with confidence, supporting better
  content segmentation and catalogue organisation for business reporting.

### ML Model - 3

In [ ]:
X_train_dense = X_train.toarray()

neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(X_train_dense)
distances, _ = neighbors.kneighbors(X_train_dense)
distances     = np.sort(distances[:, 4], axis=0)  # 4th nearest neighbour distance

plt.figure(figsize=(10, 5))
plt.plot(distances, color='steelblue')
plt.title('k-Distance Graph — Choosing DBSCAN epsilon')
plt.xlabel('Data Points (sorted)')
plt.ylabel('5th Nearest Neighbour Distance')
plt.axhline(y=1.5, color='red', linestyle='--', label='Chosen ε = 1.5')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('dbscan_kdistance.png', dpi=100)
plt.show()

# ── Fit DBSCAN ────────────────────────────────────────────────
dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_train_dense)

n_clusters_db  = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_db     = list(dbscan_labels).count(-1)

print("\n" + "=" * 55)
print("Model 3 — DBSCAN (eps=1.5, min_samples=5)")
print("=" * 55)
print(f"Clusters found : {n_clusters_db}")
print(f"Noise points   : {n_noise_db}")

if n_clusters_db >= 2:
    mask_db      = dbscan_labels != -1
    sil_db       = silhouette_score(X_train_dense[mask_db], dbscan_labels[mask_db],
                                    sample_size=min(2000, mask_db.sum()), random_state=42)
    db_db        = davies_bouldin_score(X_train_dense[mask_db], dbscan_labels[mask_db])
    print(f"Silhouette Score (excl. noise) : {sil_db:.4f}")
    print(f"Davies-Bouldin Score           : {db_db:.4f}")
else:
    sil_db = 0
    db_db  = 999
    print("Only 1 cluster found — try adjusting eps/min_samples.")

# ── DBSCAN Cluster Visualisation ──────────────────────────────
pca_db = PCA(n_components=2, random_state=42)
coords_db = pca_db.fit_transform(X_train_dense)

plt.figure(figsize=(10, 6))
unique_labels = set(dbscan_labels)
colors_db = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(sorted(unique_labels), colors_db):
    mask = dbscan_labels == label
    label_name = f'Noise' if label == -1 else f'Cluster {label}'
    marker      = 'x' if label == -1 else 'o'
    plt.scatter(coords_db[mask, 0], coords_db[mask, 1],
                c=[color], label=label_name, alpha=0.5, s=10, marker=marker)
plt.title('DBSCAN — 2D PCA Cluster Visualisation')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(loc='best', fontsize=7, markerscale=2)
plt.tight_layout()
plt.savefig('dbscan_cluster_viz.png', dpi=100)
plt.show()

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:


metrics_db = ['Silhouette Score\n(excl. noise)', 'Davies-Bouldin\n(lower=better)']
values_db  = [sil_db, db_db]
colors_db  = ['steelblue', 'tomato']

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_db, values_db, color=colors_db, edgecolor='black', width=0.4)
plt.title('DBSCAN — Evaluation Metric Scores', fontsize=14)
plt.ylabel('Score')
for bar, val in zip(bars, values_db):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('dbscan_eval_chart.png', dpi=100)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
eps_options      = [0.8, 1.0, 1.5, 2.0]
min_s_options    = [3, 5, 10]
best_db_score    = -1
best_db_params   = {}

print("\nDBSCAN Hyperparameter Search:")
print("-" * 55)

X_train_dense = X_train.toarray()

for eps, min_s in product(eps_options, min_s_options):
    db_cv = DBSCAN(eps=eps, min_samples=min_s)
    labels_cv = db_cv.fit_predict(X_train_dense)
    n_cl = len(set(labels_cv)) - (1 if -1 in labels_cv else 0)
    if n_cl < 2:
        continue
    mask = labels_cv != -1
    if mask.sum() < 50:
        continue
    score = silhouette_score(X_train_dense[mask], labels_cv[mask],
                              sample_size=min(2000, mask.sum()), random_state=42)
    noise_pct = (labels_cv == -1).mean() * 100
    print(f"  eps={eps}, min_s={min_s} → clusters={n_cl}, sil={score:.4f}, noise={noise_pct:.1f}%")
    if score > best_db_score:
        best_db_score  = score
        best_db_params = {'eps': eps, 'min_samples': min_s}

print(f"\nBest DBSCAN params : {best_db_params}  (Silhouette = {best_db_score:.4f})")

##### Which hyperparameter optimization technique have you used and why?

Manual Grid Search over eps and min_samples.
Epsilon (neighbourhood radius) and min_samples (minimum points to form a
dense region) are the two key DBSCAN parameters. We used the k-Distance plot
to narrow the search range for eps and then optimised via Silhouette Score.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

 The best eps/min_samples combination discovered through
grid search typically improves cluster quality and reduces excess noise points.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

**Primary metric: Silhouette Score**
A high silhouette score means content in the same cluster is genuinely similar
to each other and dissimilar from other clusters. This directly translates to
better content recommendations — users get titles they are likely to enjoy.

**Secondary metric: Davies-Bouldin Score**
A low DB score means clusters are compact and well-separated. For Netflix,
this ensures that different content categories (e.g., children's anime vs adult
thrillers) are cleanly separated, preventing irrelevant recommendations.

**Tertiary: Calinski-Harabasz Score**
High CH indicates densely packed, well-defined clusters — useful for business
reporting and content catalogue auditing.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

*Final Model: Tuned KMeans Clustering**

KMeans with k-means++ initialisation and the best K discovered during tuning
is selected as the final model because:
- It produced the best or near-best Silhouette and Davies-Bouldin scores.
- It is deterministic (with random_state), scalable, and fast to deploy.
- It generates fixed K clusters with interpretable centroids, making it easy
  to label clusters (e.g., "Indian Drama Movies", "Children's TV Shows").
- Agglomerative Clustering had comparable quality but cannot predict new points
  without recomputing the full hierarchy — unsuitable for live recommendation.
- DBSCAN flagged many points as noise and formed fragmented clusters,
  which is undesirable for a content catalogue recommendation engine.

# **Conclusion**

This project successfully applied unsupervised machine learning to cluster
Netflix's movie and TV show catalogue into meaningful groups.

After thorough EDA and data preprocessing — including missing value imputation,
outlier treatment, categorical encoding, text vectorisation of descriptions,
and PCA-based dimensionality reduction — three clustering algorithms were
implemented and compared: KMeans, Agglomerative Hierarchical Clustering,
and DBSCAN.

The tuned KMeans model (k-means++ initialisation, optimal K determined by
Elbow + Silhouette method) achieved the best balance of Silhouette Score,
Davies-Bouldin Score, and practical deployability.

The clustering results reveal distinct content groups based on content type,
genre, audience rating, and release era. These clusters can directly power:
  1. Personalised recommendation systems — suggesting similar titles within
     the same cluster.
  2. Content acquisition strategy — identifying underserved cluster segments.
  3. Audience targeting — matching content to viewer demographics.

The final model has been saved and validated on simulated unseen data,
confirming it is ready for deployment in a live recommendation pipeline.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***